In [1]:

from pyspark.sql.functions import *
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql.types import LongType, StringType, TimestampType, BooleanType
import ConnectionConfig as cc

In [2]:
#config
cc.setupEnvironment()
print(cc.config.sections())

Environment variables are set...
['default', 'tutorial_op', 'catchem', 'kafka']


In [3]:
#Cluster aanmaken
spark = cc.startLocalCluster("DIM_USER",4)
spark.getActiveSession()

In [4]:
#make connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")

In [5]:
#get info
user_src = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver" , cc.get_Property("driver")) \
    .option(
        "dbtable",
        "(select id, first_name, last_name, mail as email, street || ' ' || number as address from user_table) as subq"
    ) \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()\


In [7]:
user_src.createOrReplaceTempView("dimUserTemp")

df_user_dim = spark.sql("""
SELECT
  id,
  first_name,
  last_name,
  email,
  street,
    CASE WHEN t.treasure_count > 1
             THEN True ELSE False END as dedicator,
  to_timestamp('1999-01-01','yyyy-MM-dd') as scd_start,
  to_timestamp('2100-12-12','yyyy-MM-dd') as scd_end,
  True as current
FROM dimUserTemp
LEFT JOIN (
        SELECT user_id, COUNT(*) as treasure_count
        FROM treasure
        GROUP BY owner_id
    ) t
    ON d.id = t.owner_id""")

{"ts": "2025-10-05 12:49:50.259", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `treasure` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o39.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `treasure` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE 

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `treasure` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 16 pos 13;
'Project ['id, 'first_name, 'last_name, 'email, 'street, CASE WHEN ('t.treasure_count > 1) THEN true ELSE false END AS dedicator#12, to_timestamp(1999-01-01, Some(yyyy-MM-dd), TimestampType, Some(Etc/UTC), true) AS scd_start#13, to_timestamp(2100-12-12, Some(yyyy-MM-dd), TimestampType, Some(Etc/UTC), true) AS scd_end#14, true AS current#15]
+- 'Join LeftOuter, ('d.id = 't.owner_id)
   :- SubqueryAlias dimusertemp
   :  +- View (`dimUserTemp`, [id#0, first_name#1, last_name#2, email#3, address#4])
   :     +- Relation [id#0,first_name#1,last_name#2,email#3,address#4] JDBCRelation((select 

In [8]:
#opslagen tabel
user_src.write.format("delta").mode("overwrite").save("delta/USER_DIM")

In [47]:
spark.stop()

In [16]:
# Tabel 'treasure' laden via JDBC
treasure_src = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver", cc.get_Property("driver")) \
    .option("dbtable", "treasure") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

# Enkele rijen bekijken
treasure_src.show(5)  # toont de eerste 5 rijen


+--------------------+----------+-------+--------------------+--------------------+
|                  id|difficulty|terrain|        city_city_id|            owner_id|
+--------------------+----------+-------+--------------------+--------------------+
|[00 00 3E 2C B1 4...|         3|      1|[62 34 F0 0E 0E 9...|[B3 10 59 70 E9 6...|
|[00 03 72 3C 7C C...|         3|      2|[59 07 42 E8 55 1...|[75 AA F7 4A 33 C...|
|[00 04 39 A0 98 7...|         4|      4|[DB 39 44 FB 1D 0...|[8B CC 87 A3 8C 6...|
|[00 04 62 1B 29 E...|         1|      2|[66 97 B3 4B 1F 1...|[11 C4 C7 96 95 6...|
|[00 05 1E A2 05 8...|         0|      2|[38 EC 6D 3C 5A 1...|[DD AD C2 FE 29 5...|
+--------------------+----------+-------+--------------------+--------------------+
only showing top 5 rows
